# Dataset Creation from 1000+ Resumes

This section describes the process of creating a dataset from over 1000 resumes. The dataset will be used for analysis, such as extracting key information like skills, experience, and education.

In [ ]:
# Install Google Chrome manually via .deb file
# !wget https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
# !apt-get install -y ./google-chrome-stable_current_amd64.deb

# !pip install pandas beautifulsoup4 tqdm selenium webdriver-manager

In [3]:
!pip install selenium webdriver-manager beautifulsoup4

  Using cached selenium-4.38.0-py3-none-any.whl.metadata (7.5 kB)
  Using cached webdriver_manager-4.0.2-py2.py3-none-any.whl.metadata (12 kB)
  Using cached trio-0.32.0-py3-none-any.whl.metadata (8.5 kB)
  Using cached trio_websocket-0.12.2-py3-none-any.whl.metadata (5.1 kB)
  Using cached websocket_client-1.9.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached sortedcontainers-2.4.0-py2.py3-none-any.whl.metadata (10 kB)
  Using cached outcome-1.3.0.post0-py2.py3-none-any.whl.metadata (2.6 kB)
  Using cached wsproto-1.3.2-py3-none-any.whl.metadata (5.2 kB)
  Using cached python_dotenv-1.2.1-py3-none-any.whl.metadata (25 kB)
  Using cached soupsieve-2.8-py3-none-any.whl.metadata (4.6 kB)
Using cached selenium-4.38.0-py3-none-any.whl (9.7 MB)
Using cached trio-0.32.0-py3-none-any.whl (512 kB)
Using cached trio_websocket-0.12.2-py3-none-any.whl (21 kB)
Using cached websocket_client-1.9.0-py3-none-any.whl (82 kB)
Using cached webdriver_manager-4.0.2-py2.py3-none-any.whl (27 kB)
Using cach

In [4]:
import os
import json
import re
import pandas as pd
import numpy as np

import time
from datetime import datetime
from tqdm import tqdm
import logging
import requests

from pathlib import Path
from typing import Optional, Dict, Any, Tuple


from PyPDF2 import PdfReader
from PyPDF2.errors import PdfReadError
from docx import Document
from docx.opc.constants import RELATIONSHIP_TYPE as RT
from docx.opc.exceptions import PackageNotFoundError

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup



In [ ]:
try:
    import win32com.client as win32  # type: ignore
    import pywintypes  # type: ignore
except ImportError:
    win32 = None
    pywintypes = None

RECOVERABLE_EXTRACTION_ERRORS = (
    ValueError,
    OSError,
    PdfReadError,
    PackageNotFoundError,
    KeyError,
    RuntimeError,
)
# if pywintypes is not None and hasattr(pywintypes, "com_error"):
#     RECOVERABLE_EXTRACTION_ERRORS = RECOVERABLE_EXTRACTION_ERRORS + (
#         pywintypes.com_error,  # type: ignore[attr-defined]
#     )    

In [ ]:
SUPPORTED_EXTENSIONS = {".pdf", ".docx", ".doc"}
DEFAULT_INPUT_PATH = Path(r"C:\Users\Abhinav\Documents\Repo\Colab-proj-2\data\resumes")
DEFAULT_OUTPUT_PATH = Path("dataset/resume_text.jsonl")

In [ ]:
class WordAutomationClient:
    """Thin wrapper around Word COM automation to read legacy .doc files."""

    def __init__(self) -> None:
        if win32 is None:
            raise ImportError("pywin32 is required for .doc support on Windows")
        self._word = win32.Dispatch("Word.Application")
        self._word.Visible = False

    def close(self) -> None:
        if self._word is not None:
            self._word.Quit()
            self._word = None

    def __enter__(self) -> "WordAutomationClient":
        return self

    def __exit__(self, exc_type, exc, exc_tb) -> None:  # type: ignore[override]
        self.close()

    def extract_doc(self, file_path: Path) -> tuple[str, bool]:
        if self._word is None:
            raise RuntimeError("Word automation client is closed")
        document = self._word.Documents.Open(str(file_path))
        try:
            text = document.Content.Text
            contains_images = bool(document.InlineShapes.Count or document.Shapes.Count)
        finally:
            document.Close(False)
        return text.strip(), contains_images


def extract_text_from_pdf(file_path: Path) -> str:
    """Extract text from a PDF file."""
    reader = PdfReader(file_path)
    text = ""
    for page in reader.pages:
        text += (page.extract_text() or "") + "\n"
    return text.strip()


def extract_text_from_docx(file_path: Path) -> str:
    """Extract text from a DOCX file."""
    doc = Document(file_path)
    text = ""
    for para in doc.paragraphs:
        text += para.text + "\n"
    return text.strip()


def extract_text(file_path: Path) -> str:
    """Extract text from a document file based on its extension."""
    suffix = file_path.suffix.lower()
    if suffix == ".pdf":
        return extract_text_from_pdf(file_path)
    if suffix == ".docx":
        return extract_text_from_docx(file_path)
    raise ValueError(f"Unsupported file type: {file_path.suffix}")


def _xobject_dict_contains_images(x_objects) -> bool:
    if not x_objects:
        return False
    # Dereference x_objects if it's an indirect object
    try:
        x_objects = x_objects.get_object()
    except AttributeError:
        pass
    if not hasattr(x_objects, "values"):
        return False
    for obj in x_objects.values():
        try:
            x_obj = obj.get_object()
        except AttributeError:
            x_obj = obj
        subtype = x_obj.get("/Subtype")
        if subtype == "/Image":
            return True
        if subtype == "/Form":
            child_resources = x_obj.get("/Resources")
            child_x_objects = None
            if child_resources:
                # Dereference indirect objects
                try:
                    child_resources = child_resources.get_object()
                except AttributeError:
                    pass
                if hasattr(child_resources, "get"):
                    child_x_objects = child_resources.get("/XObject")
            if _xobject_dict_contains_images(child_x_objects):
                return True
    return False


def pdf_has_images(file_path: Path) -> bool:
    reader = PdfReader(file_path)
    for page in reader.pages:
        if hasattr(page, "images") and page.images:
            return True
        resources = page.get("/Resources")
        if not resources:
            continue
        # Dereference indirect objects
        try:
            resources = resources.get_object()
        except AttributeError:
            pass
        if not hasattr(resources, "get"):
            continue
        x_objects = resources.get("/XObject")
        if _xobject_dict_contains_images(x_objects):
            return True
    return False


def docx_has_images(file_path: Path) -> bool:
    doc = Document(file_path)
    return any(rel.reltype == RT.IMAGE for rel in doc.part.rels.values())


def has_images(file_path: Path) -> bool:
    suffix = file_path.suffix.lower()
    if suffix == ".pdf":
        return pdf_has_images(file_path)
    if suffix == ".docx":
        return docx_has_images(file_path)
    raise ValueError(f"Unsupported file type: {file_path.suffix}")


def collect_documents(target: Path) -> List[Path]:
    if target.is_file():
        return [target]
    if target.is_dir():
        return sorted(
            [f for f in target.rglob("*") if f.suffix.lower() in SUPPORTED_EXTENSIONS]
        )
    raise FileNotFoundError(f"'{target}' is not a valid file or directory.")


def process_document(
    file_path: Path, word_client: Optional["WordAutomationClient"]
) -> Dict[str, str | bool]:
    suffix = file_path.suffix.lower()
    if suffix == ".doc":
        if word_client is None:
            raise RuntimeError(
                ".doc support requires Microsoft Word and pywin32; both appear unavailable."
            )
        resume_text, contains_images = word_client.extract_doc(file_path)
    else:
        resume_text = extract_text(file_path)
        contains_images = has_images(file_path)

    return {
        "filename": file_path.name,
        "filetype": suffix,
        "resume_text": resume_text,
        "has_images": contains_images,
    }


def main() -> None:
    # Update these paths to control which resumes are processed and where JSONL output lands.
    input_path = DEFAULT_INPUT_PATH
    output_path = DEFAULT_OUTPUT_PATH

    try:
        documents = collect_documents(input_path)
    except FileNotFoundError as exc:
        print(exc)
        return

    if not documents:
        print(f"No PDF/DOC/DOCX files found under '{input_path}'.")
        return

    needs_word = any(path.suffix.lower() == ".doc" for path in documents)
    word_client: Optional[WordAutomationClient] = None
    if needs_word:
        try:
            word_client = WordAutomationClient()
        except ImportError as exc:
            print(f"Cannot open .doc files: {exc}")
            return

    processed_count = 0
    output_path.parent.mkdir(parents=True, exist_ok=True)
    try:
        with output_path.open("w", encoding="utf-8") as fh:
            for doc_path in documents:
                try:
                    record = process_document(doc_path, word_client)
                except RECOVERABLE_EXTRACTION_ERRORS as exc:
                    print(f"Skipping {doc_path}: {exc}")
                    continue
                fh.write(json.dumps(record, ensure_ascii=False) + "\n")
                processed_count += 1
                print(f"Processed {doc_path.name}")
    finally:
        if word_client is not None:
            word_client.close()

    if processed_count:
        print(
            f"Wrote metadata for {processed_count} file(s) to '{output_path.as_posix()}'"
        )
    else:
        print("No files were successfully processed; see logs above for details.")



In [ ]:
# running the main function
main()

# Building a Job Scraper

This section outlines the development of a job scraper to collect job postings from various sources. The scraper will extract relevant information such as job titles, descriptions, requirements, and company details to complement the resume dataset for analysis or matching purposes.

In [ ]:
# import requests

try:
    response = requests.get('https://api64.ipify.org?format=json')
    response.raise_for_status() # Raise an exception for HTTP errors
    ip_data = response.json()
    ip_address = ip_data.get('ip')
    if ip_address:
        print(f"Your IP Address is: {ip_address}")
    else:
        print("Could not retrieve IP address from the service.")
except requests.exceptions.RequestException as e:
    print(f"Error fetching IP address: {e}")
except ValueError:
    print("Error decoding JSON response.")

In [ ]:
# from selenium import webdriver
# from selenium.webdriver.chrome.service import Service
# from webdriver_manager.chrome import ChromeDriverManager
# from bs4 import BeautifulSoup

# import pandas as pd
# import time
# import logging
# import os
# from datetime import datetime
# from tqdm import tqdm

In [ ]:
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [ ]:
class JobScraper:
    """Handles web scraping of job descriptions from various job posting websites."""

    def __init__(self, headless=True, wait_time=5, delay=2):
        """
        Initialize the job description scraper.

        Args:
            headless (bool): Run browser in headless mode
            wait_time (int): Time to wait for page to load (seconds)
            delay (int): Delay between requests (seconds)
        """
        self.headless = headless
        self.wait_time = wait_time
        self.delay = delay
        self.driver = None

    def setup_driver(self):
        """Set up Chrome WebDriver with appropriate options."""
        options = webdriver.ChromeOptions()
        if self.headless:
            options.add_argument('--headless')
        options.add_argument('--no-sandbox')
        options.add_argument('--disable-dev-shm-usage')
        options.add_argument('--disable-gpu')
        options.add_argument('--window-size=1920,1080')

        self.driver = webdriver.Chrome(
            service=Service(ChromeDriverManager().install()),
            options=options
        )
        return self.driver

    def close_driver(self):
        """Close the WebDriver if it exists."""
        if self.driver:
            self.driver.quit()
            self.driver = None

    def scrape_job(self, job_url):
        """
        Extract job description from a job posting URL.

        Args:
            job_url (str): URL of the job posting

        Returns:
            tuple: (job_description, status)
        """
        if not self.driver:
            self.setup_driver()

        try:
            self.driver.get(job_url)
            time.sleep(self.wait_time)

            soup = BeautifulSoup(self.driver.page_source, 'html.parser')

            body_text = soup.body.get_text() if soup.body else ""

            # Extract the job description part
            # Assuming it starts after job details like Full-time, Onsite, etc.
            lines = body_text.split('\n')
            desc_start = False
            description = []

            for line in lines:
                line = line.strip()
                if any(keyword in line for keyword in ['Full-time', 'Onsite', 'Remote', 'Hybrid', 'Part-time']):
                    desc_start = True
                if desc_start and line:
                    description.append(line)

            job_text = ' '.join(description)

            if len(job_text) < 100:
                return None, "Description too short (< 100 chars)"

            # Truncate very long descriptions
            if len(job_text) > 10000:
                job_text = job_text[:10000] + "..."

            return job_text, "Success"

        except Exception as e:
            logger.error(f"Error scraping URL {job_url}: {str(e)}")
            return None, f"Error: {str(e)}"

    def scrape_from_csv(self, input_csv, output_csv=None):
        """Scrape jobs from CSV file with URLs."""

        if not os.path.exists(input_csv):
            logger.error(f"Input CSV file not found: {input_csv}")
            return None

        # Read input CSV
        try:
            df = pd.read_csv(input_csv)
        except Exception as e:
            logger.error(f"Error reading CSV: {e}")
            return None

        # Find URL column - look for 'Apply', 'url', or 'link' columns
        url_column = None

        # Priority order: Apply > URL > Link
        column_priorities = ['apply', 'url', 'link']

        for priority_col in column_priorities:
            for col in df.columns:
                if priority_col in col.lower():
                    url_column = col
                    break
            if url_column:
                break

        if url_column is None:
            logger.error("No URL column found in CSV. Expected column with 'Apply', 'URL', or 'Link' in name.")
            logger.info(f"Available columns: {list(df.columns)}")
            return None

        logger.info(f"Found {len(df)} URLs to scrape in column '{url_column}'")

        # Create output filename if not provided
        if output_csv is None:
            timestamp = datetime.now().strftime("%H%M%S%m%d%Y")
            output_csv = f"data/scraped_jobs_{timestamp}.csv"
            os.makedirs("data", exist_ok=True)

        # Create CSV file with headers if it doesn't exist
        file_exists = os.path.exists(output_csv)
        if not file_exists:
            with open(output_csv, 'w', encoding='utf-8', newline='') as f:
                f.write('URL,Job Description,Scrape Status\n')

        # Scrape each URL
        successful = 0
        failed = 0

        try:
            with tqdm(total=len(df), desc="Scraping jobs") as pbar:
                for idx, row in df.iterrows():
                    job_url = row[url_column]

                    if pd.isna(job_url) or not job_url.strip():
                        # Save immediately to CSV
                        result_df = pd.DataFrame([[job_url, "", "Empty URL"]],
                                                columns=['URL', 'Job Description', 'Scrape Status'])
                        result_df.to_csv(output_csv, mode='a', header=False, index=False, encoding='utf-8')
                        failed += 1
                        pbar.update(1)
                        continue

                    # Scrape job
                    description, status = self.scrape_job(job_url)

                    # Save immediately to CSV after each scrape
                    result_df = pd.DataFrame([[
                        job_url,
                        description if description else "",
                        status
                    ]], columns=['URL', 'Job Description', 'Scrape Status'])

                    result_df.to_csv(output_csv, mode='a', header=False, index=False, encoding='utf-8')

                    if status == "Success":
                        successful += 1
                    else:
                        failed += 1

                    pbar.set_postfix({
                        'Success': successful,
                        'Failed': failed,
                        'Rate': f"{successful/(successful+failed)*100:.1f}%" if (successful+failed) > 0 else "0%"
                    })

                    # Delay between requests
                    time.sleep(self.delay)
                    pbar.update(1)

            logger.info(f"Results saved to {output_csv}")
            logger.info(f"Summary: {successful} successful, {failed} failed")
            return output_csv,result_df

        except Exception as e:
            logger.error(f"Error during scraping: {e}")
            logger.info(f"Partial results saved to {output_csv}")
            logger.info(f"Summary before error: {successful} successful, {failed} failed")
            return output_csv,result_df
        finally:
            self.close_driver()

# Cleaning and Combining the Scraped Job and Resume Data
This section focuses on processing and integrating the scraped job descriptions with the resume dataset. Key steps include:

- Loading the cleaned scraped job data from CSV and the resume text data from JSONL.
- Filtering and cleaning the resume data to remove entries with images and normalize text.
- Randomly assigning job descriptions to resumes to create a balanced dataset for potential matching or analysis tasks.
- Saving the combined dataset in multiple formats (CSV, Parquet, JSONL) for flexibility and efficiency.
- Comparing file sizes and data integrity across formats to ensure consistency.
- The resulting dataset pairs each resume with a job description, facilitating downstream tasks like resume-job matching or machine learning model training. Note that Parquet offers the most compact storage, while JSONL preserves structure for streaming.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

input_csv = "dataset/merged_csv.csv"
output_folder = 'dataset/scrape_job/'

In [ ]:


# Ensure the Google Drive output folder exists
os.makedirs(output_folder, exist_ok=True)

# Updated output_csv to save to Google Drive
timestamp = datetime.now().strftime("%H%M%S%m%d%Y")
output_csv = os.path.join(output_folder, f"scraped_jobs_{timestamp}.csv")

scraper = JobScraper()
print("Starting job scraping...")
result_file = scraper.scrape_from_csv(input_csv, output_csv)

if result_file:
    print(f"Scraping complete! Results saved to {result_file}")
else:
    print("Scraping failed!")


In [ ]:

df_jobs = pd.read_csv('dataset/cleaned_scraped_jobs_21390711212025.csv')
df_resumes = pd.read_json('dataset/resume_text.jsonl', lines=True)

In [ ]:
df_resumes = df_resumes[~((df_resumes['has_images'] == True))]


# Function to clean text by removing non-ASCII characters and extra whitespace
def clean_text(text):
    # Remove non-ASCII characters
    text = re.sub(r'[^\x00-\x7F]+', '', text)
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Apply cleaning to resume_text and filename columns
df_resumes['resume_text'] = df_resumes['resume_text'].apply(clean_text)
df_resumes['filename'] = df_resumes['filename'].apply(clean_text)

# Optionally, filter out rows where resume_text is empty or too short after cleaning
df_resumes = df_resumes[df_resumes['resume_text'].str.len() > 10]  # Example: keep rows with more than 10 characters

print(f"Resumes DataFrame shape after cleaning: {df_resumes.shape}")

# Clean the resume_text column by replacing any sequence of whitespace (including newlines, tabs, etc.) with a single space
df_resumes['resume_text'] = df_resumes['resume_text'].str.replace(r'[^\w\s]', '', regex=True).str.replace(r'\s+', ' ', regex=True)


In [13]:
df_resumes.head()
df_resumes.info()

df_jobs.head()
df_jobs.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 249 entries, 0 to 248
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   URL              249 non-null    object
 1   Job Description  249 non-null    object
 2   Scrape Status    249 non-null    object
dtypes: object(3)
memory usage: 6.0+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 249 entries, 0 to 248
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   URL              249 non-null    object
 1   Job Description  249 non-null    object
 2   Scrape Status    249 non-null    object
dtypes: object(3)
memory usage: 6.0+ KB


In [ ]:


# Get all job descriptions
job_descriptions = df_jobs['Job Description'].values

# Calculate how many times each job description should be used (approximately)
n_resumes = len(df_resumes)
n_jobs = len(job_descriptions)

# Create an array of job indices that will be evenly distributed
# Repeat the job indices enough times to cover all resumes
repeats = (n_resumes // n_jobs) + 1
job_indices = np.tile(np.arange(n_jobs), repeats)[:n_resumes]

# Shuffle to randomize the assignment
np.random.seed(42)  # For reproducibility
np.random.shuffle(job_indices)

# Assign job descriptions to resumes
df_resumes['job_description'] = job_descriptions[job_indices]

# Verify the distribution
print(f"Total resumes: {len(df_resumes)}")
print(f"Unique job descriptions: {df_resumes['job_description'].nunique()}")
print(f"\nDistribution of job descriptions (counts):")
print(df_resumes['job_description'].value_counts().describe())
print(f"\nMin count: {df_resumes['job_description'].value_counts().min()}")
print(f"Max count: {df_resumes['job_description'].value_counts().max()}")
print(f"\nDataFrame info:")
df_resumes.info()

Total resumes: 1565
Unique job descriptions: 245

Distribution of job descriptions (counts):
count    245.000000
mean       6.387755
std        0.882812
min        6.000000
25%        6.000000
50%        6.000000
75%        7.000000
max       13.000000
Name: count, dtype: float64

Min count: 6
Max count: 13

DataFrame info:
<class 'pandas.core.frame.DataFrame'>
Index: 1565 entries, 0 to 2126
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   filename         1565 non-null   object
 1   filetype         1565 non-null   object
 2   resume_text      1565 non-null   object
 3   has_images       1565 non-null   bool  
 4   job_description  1565 non-null   object
dtypes: bool(1), object(4)
memory usage: 62.7+ KB


In [ ]:
df_resumes.to_csv(os.path.join(DATA_DIR, 'resume_dataset.csv'), index=False)
df_resumes.to_parquet(os.path.join(DATA_DIR, 'resume_dataset.parquet'), engine='pyarrow', compression='snappy')
df_resumes.to_json(os.path.join(DATA_DIR, 'resume_dataset.jsonl'), orient='records', lines=True)

In [ ]:
# Read the saved files and compare them


DATA_DIR = ".//dataset"

# Read all three formats
df_csv = pd.read_csv(os.path.join(DATA_DIR, 'resume_dataset.csv'))
df_parquet = pd.read_parquet(os.path.join(DATA_DIR, 'resume_dataset.parquet'), engine='pyarrow')
df_jsonl = pd.read_json(os.path.join(DATA_DIR, 'resume_dataset.jsonl'), lines=True)

# Compare shapes
print("=== Shape Comparison ===")
print(f"CSV:     {df_csv.shape}")
print(f"Parquet: {df_parquet.shape}")
print(f"JSONL:   {df_jsonl.shape}")

# Compare columns
print("\n=== Columns Comparison ===")
print(f"CSV columns:     {list(df_csv.columns)}")
print(f"Parquet columns: {list(df_parquet.columns)}")
print(f"JSONL columns:   {list(df_jsonl.columns)}")

# Compare data types
print("\n=== Data Types Comparison ===")
print("CSV dtypes:")
print(df_csv.dtypes)
print("\nParquet dtypes:")
print(df_parquet.dtypes)
print("\nJSONL dtypes:")
print(df_jsonl.dtypes)

# Compare file sizes
print("\n=== File Size Comparison ===")
csv_size = os.path.getsize(os.path.join(DATA_DIR, 'resume_dataset.csv'))
parquet_size = os.path.getsize(os.path.join(DATA_DIR, 'resume_dataset.parquet'))
jsonl_size = os.path.getsize(os.path.join(DATA_DIR, 'resume_dataset.jsonl'))

print(f"CSV:     {csv_size / (1024*1024):.2f} MB")
print(f"Parquet: {parquet_size / (1024*1024):.2f} MB")
print(f"JSONL:   {jsonl_size / (1024*1024):.2f} MB")

# Check if data is identical
print("\n=== Data Equality Check ===")
print(f"CSV == Parquet: {df_csv.equals(df_parquet)}")
print(f"CSV == JSONL:   {df_csv.equals(df_jsonl)}")
print(f"Parquet == JSONL: {df_parquet.equals(df_jsonl)}")

# Show sample data
print("\n=== Sample Data (first 2 rows) ===")
df_csv.head(2)

=== Shape Comparison ===
CSV:     (1565, 5)
Parquet: (1565, 5)
JSONL:   (1565, 5)

=== Columns Comparison ===
CSV columns:     ['filename', 'filetype', 'resume_text', 'has_images', 'job_description']
Parquet columns: ['filename', 'filetype', 'resume_text', 'has_images', 'job_description']
JSONL columns:   ['filename', 'filetype', 'resume_text', 'has_images', 'job_description']

=== Data Types Comparison ===
CSV dtypes:
filename           object
filetype           object
resume_text        object
has_images           bool
job_description    object
dtype: object

Parquet dtypes:
filename           object
filetype           object
resume_text        object
has_images           bool
job_description    object
dtype: object

JSONL dtypes:
filename           object
filetype           object
resume_text        object
has_images           bool
job_description    object
dtype: object

=== File Size Comparison ===
CSV:     25.36 MB
Parquet: 9.48 MB
JSONL:   25.50 MB

=== Data Equality Check ===
C

,filename,filetype,resume_text,has_images,job_description
0,00_Willie_Ellis_Go_Python.docx,.docx,Willie Ellis Senior Software Engineer Buffalo ...,False,Hanger/Textiles jobs in United StatesOverviewC...
1,10272022 Resume.docx,.docx,SUMMARY Leverage my skills education and exper...,False,Production Associate - Garment Hanger/Inspecto...


In [18]:
DATA_DIR = ".//dataset"
df = pd.read_parquet(os.path.join(DATA_DIR, 'resume_dataset.parquet'), engine='pyarrow')

# Ollama Resume Generator

This section uses a local Ollama model to generate tailored resumes based on the resume text and job descriptions.

In [19]:
# JSON Schema for tailored resume output
RESUME_SCHEMA = '''{"$schema":"http://json-schema.org/draft-04/schema#","type":"object","properties":{"personal_information":{"type":"object","properties":{"name":{"type":"string"},"email":{"type":"string"},"phone":{"type":"string"},"location":{"type":"string"},"socials":{"type":"array","items":[{"type":"object","properties":{"name":{"type":"string"},"link":{"type":"string"}},"required":["name","link"]}]}},"required":["name","email","phone","location"]},"summary":{"type":"string"},"experiences":{"type":"array","items":[{"type":"object","properties":{"designation":{"type":"string"},"companyName":{"type":"string"},"location":{"type":"string"},"start_date":{"type":"string"},"end_date":{"type":"string"},"points":{"type":"array","items":[{"type":"string"}]}},"required":["designation","companyName","location","start_date"]}]},"education":{"type":"array","items":[{"type":"object","properties":{"institution":{"type":"string"},"degree":{"type":"string"},"location":{"type":"string"},"start_date":{"type":"string"},"end_date":{"type":"string"},"gpa":{"type":"string"}},"required":["institution","degree","location","start_date","gpa"]}]},"skills":{"type":"array","items":[{"type":"object","properties":{"name":{"type":"string"},"data":{"type":"array","items":[{"type":"string"}]}},"required":["name","data"]}]},"projects":{"type":"array","items":[{"type":"object","properties":{"projectName":{"type":"string"},"caption":{"type":"string"},"location":{"type":"string"},"start_date":{"type":"string"},"end_date":{"type":"string"},"url":{"type":"string"},"projectDetails":{"type":"array","items":[{"type":"string"}]},"externalSources":{"type":"array","items":[{"type":"object","properties":{"name":{"type":"string"},"link":{"type":"string"}},"required":["name","link"]}]},"technologiesUsed":{"type":"array","items":[{"type":"string"}]}},"required":["projectName","location","projectDetails"]}]},"certifications":{"type":"array","items":[{"type":"object","properties":{"name":{"type":"string"},"issuing_organization":{"type":"string"},"issue_date":{"type":"string"},"expiration_date":{"type":"string"},"credential_id":{"type":"string"},"url":{"type":"string"}},"required":["name","issuing_organization","issue_date","expiration_date","credential_id","url"]}]},"awards":{"type":"array","items":[{"type":"object","properties":{"name":{"type":"string"},"type":{"type":"string"},"location":{"type":"string"},"date":{"type":"string"},"description":{"type":"string"}},"required":["name","type","location","date","description"]}]},"extracurricular_achievements":{"type":"array","items":[{"type":"object","properties":{"name":{"type":"string"},"type":{"type":"string"},"location":{"type":"string"},"date":{"type":"string"},"description":{"type":"string"}},"required":["name","type","location","date","description"]}]},"languages":{"type":"array","items":[{"type":"object","properties":{"language":{"type":"string"},"proficiency":{"type":"string"}},"required":["language","proficiency"]}]}},"required":["personal_information","education","skills","extracurricular_achievements"]}'''

# Load prompt template
with open('dataset/prompt.txt', 'r', encoding='utf-8') as f:
    PROMPT_TEMPLATE = f.read()

print("Prompt template loaded successfully")
print(f"Schema defined:\n{RESUME_SCHEMA[:200]}...")
print(f"Prompt template preview:\n{PROMPT_TEMPLATE[:500]}...")

Prompt template loaded successfully
Schema defined:
{"$schema":"http://json-schema.org/draft-04/schema#","type":"object","properties":{"personal_information":{"type":"object","properties":{"name":{"type":"string"},"email":{"type":"string"},"phone":{"ty...
Prompt template preview:
You create a tailored resume based on the job description.

Your task:
1. Read the RESUME_TEXT.
2. Read the JOB_DESCRIPTION.
3. Use only the information inside these two.
4. Follow the SCHEMA exactly.
5. Write a tailored resume in JSON using the SCHEMA.
6. Do not output anything outside the JSON.
7. If a field is missing in the resume, write a short, safe placeholder that fits the job.


RESUME_TEXT:
"""
<<PASTE RESUME HERE>>
"""

JOB_DESCRIPTION:
"""
<<PASTE JD HERE>>
"""

SCHEMA:
"""
<<PASTE J...


In [ ]:


class OllamaResumeGenerator:
    """Class to generate tailored resumes using local Ollama model (supports thinking models)."""
    
    def __init__(
        self,
        model_name: str = "qwen3:8b",
        base_url: str = "http://localhost:11434",
        output_path: str = "dataset/tailored_resumes",  # Base name without extension
        timeout: int = 300,  # Increased for thinking models
        max_retries: int = 3,
        enable_thinking: bool = True  # Enable thinking mode for supported models
    ):
        """
        Initialize the Ollama Resume Generator.
        
        Args:
            model_name: Name of the Ollama model to use
            base_url: Base URL for the Ollama API
            output_path: Base path for output file (timestamp will be added)
            timeout: Request timeout in seconds (higher for thinking models)
            max_retries: Maximum number of retries on failure
            enable_thinking: Whether to enable thinking mode (adds /think suffix)
        """
        self.model_name = model_name
        self.base_url = base_url
        self.api_url = f"{base_url}/api/generate"
        self.timeout = timeout
        self.max_retries = max_retries
        self.enable_thinking = enable_thinking
        
        # Add timestamp to output filename
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        self.output_path = Path(f"{output_path}_{timestamp}.jsonl")
        
        # Ensure output directory exists
        self.output_path.parent.mkdir(parents=True, exist_ok=True)
        
        # Store session start time
        self.session_start = datetime.now().isoformat()
        
    def _build_prompt(self, resume_text: str, job_description: str) -> str:
        """Build the prompt by filling in the template."""
        prompt = PROMPT_TEMPLATE.replace("<<PASTE RESUME HERE>>", resume_text)
        prompt = prompt.replace("<<PASTE JD HERE>>", job_description)
        prompt = prompt.replace("<<PASTE JSON SCHEMA HERE>>", RESUME_SCHEMA)
        return prompt
    
    def _extract_thinking_and_response(self, response: str) -> Tuple[Optional[str], str]:
        """
        Extract thinking content and actual response from thinking model output.
        
        Returns:
            Tuple of (thinking_content, actual_response)
        """
        if not response:
            return None, ""
        
        thinking_content = None
        actual_response = response
        
        # Pattern to match <think>...</think> blocks
        think_pattern = r'<think>(.*?)</think>'
        think_match = re.search(think_pattern, response, re.DOTALL)
        
        if think_match:
            thinking_content = think_match.group(1).strip()
            # Remove the thinking block from response
            actual_response = re.sub(think_pattern, '', response, flags=re.DOTALL).strip()
        
        return thinking_content, actual_response
    
    def _call_ollama(self, prompt: str) -> Tuple[Optional[str], Optional[str]]:
        """
        Make a request to the Ollama API.
        
        Returns:
            Tuple of (response, thinking_content)
        """
        payload = {
            "model": self.model_name,
            "prompt": prompt,
            "stream": False,
            # "options": {
            #     "temperature": 0.7,
            #     "num_predict": 4096  # Increased for thinking models
            # }
        }
        
        for attempt in range(self.max_retries):
            try:
                response = requests.post(
                    self.api_url,
                    json=payload,
                    timeout=self.timeout
                )
                response.raise_for_status()
                result = response.json()
                raw_response = result.get("response", "")
                
                # Extract thinking and actual response
                thinking, actual_response = self._extract_thinking_and_response(raw_response)
                
                return actual_response, thinking
                
            except requests.exceptions.Timeout:
                print(f"Timeout on attempt {attempt + 1}/{self.max_retries}")
            except requests.exceptions.RequestException as e:
                print(f"Request error on attempt {attempt + 1}/{self.max_retries}: {e}")
            
            if attempt < self.max_retries - 1:
                time.sleep(2 ** attempt)  # Exponential backoff
        
        return None, None
    
    def _extract_json(self, response: str) -> Optional[Dict[str, Any]]:
        """Extract JSON from the model response."""
        if not response:
            return None
        
        # Try to find JSON in the response
        response = response.strip()
        
        # Try direct parsing first
        try:
            return json.loads(response)
        except json.JSONDecodeError:
            pass
        
        # Try to extract JSON from markdown code blocks
        if "```json" in response:
            start = response.find("```json") + 7
            end = response.find("```", start)
            if end > start:
                try:
                    return json.loads(response[start:end].strip())
                except json.JSONDecodeError:
                    pass
        
        # Try generic code blocks
        if "```" in response:
            start = response.find("```") + 3
            # Skip language identifier if present
            newline_pos = response.find("\n", start)
            if newline_pos > start:
                start = newline_pos + 1
            end = response.find("```", start)
            if end > start:
                try:
                    return json.loads(response[start:end].strip())
                except json.JSONDecodeError:
                    pass
        
        # Try to extract JSON between curly braces
        start = response.find("{")
        end = response.rfind("}") + 1
        if start >= 0 and end > start:
            try:
                return json.loads(response[start:end])
            except json.JSONDecodeError:
                pass
        
        return None
    
    def generate_single(self, resume_text: str, job_description: str, filename: str) -> Dict[str, Any]:
        """Generate a tailored resume for a single resume-job pair."""
        start_time = datetime.now()
        prompt = self._build_prompt(resume_text, job_description)
        response, thinking_content = self._call_ollama(prompt)
        end_time = datetime.now()
        
        result = {
            "filename": filename,
            "original_resume": resume_text[:500] + "..." if len(resume_text) > 500 else resume_text,
            "job_description": job_description[:500] + "..." if len(job_description) > 500 else job_description,
            "status": "success",
            "tailored_resume": None,
            "raw_response": None,
            "thinking_content": thinking_content,  # Store the model's reasoning
            "timestamp": end_time.isoformat(),
            "processing_time_seconds": (end_time - start_time).total_seconds()
        }
        
        if response:
            parsed_json = self._extract_json(response)
            if parsed_json:
                result["tailored_resume"] = parsed_json
            else:
                result["status"] = "json_parse_error"
                result["raw_response"] = response[:2000] if len(response) > 2000 else response
        else:
            result["status"] = "api_error"
        
        return result
    
    def process_dataframe(
        self,
        df: pd.DataFrame,
        resume_col: str = "resume_text",
        job_col: str = "job_description",
        filename_col: str = "filename",
        start_idx: int = 0,
        end_idx: Optional[int] = None,
        save_every: int = 10
    ) -> None:
        """
        Process a DataFrame and generate tailored resumes.
        
        Args:
            df: DataFrame with resume and job description columns
            resume_col: Name of the resume text column
            job_col: Name of the job description column
            filename_col: Name of the filename column
            start_idx: Starting index for processing
            end_idx: Ending index for processing (None = process all)
            save_every: Save progress after every N records
        """
        if end_idx is None:
            end_idx = len(df)
        
        df_subset = df.iloc[start_idx:end_idx]
        
        successful = 0
        failed = 0
        
        batch_start_time = datetime.now()
        print(f"{'='*50}")
        print(f"Starting processing at: {batch_start_time.isoformat()}")
        print(f"Model: {self.model_name}")
        print(f"Thinking mode: {'Enabled' if self.enable_thinking else 'Disabled'}")
        print(f"Output file: {self.output_path}")
        print(f"Processing records {start_idx} to {end_idx} ({len(df_subset)} total)")
        print(f"{'='*50}\n")
        
        # Open file in append mode
        with open(self.output_path, 'a', encoding='utf-8') as fh:
            for idx, row in tqdm(df_subset.iterrows(), total=len(df_subset), desc="Generating resumes"):
                resume_text = row[resume_col]
                job_description = row[job_col]
                filename = row[filename_col]
                
                result = self.generate_single(resume_text, job_description, filename)
                result["original_index"] = idx
                result["session_start"] = self.session_start
                result["model_used"] = self.model_name
                
                # Write to file immediately
                fh.write(json.dumps(result, ensure_ascii=False) + "\n")
                
                if result["status"] == "success":
                    successful += 1
                else:
                    failed += 1
                
                # Flush periodically
                if (successful + failed) % save_every == 0:
                    fh.flush()
        
        batch_end_time = datetime.now()
        total_time = (batch_end_time - batch_start_time).total_seconds()
        
        print(f"\n{'='*50}")
        print(f"Processing complete!")
        print(f"Model:      {self.model_name}")
        print(f"Started:    {batch_start_time.isoformat()}")
        print(f"Finished:   {batch_end_time.isoformat()}")
        print(f"Total time: {total_time:.2f} seconds ({total_time/60:.2f} minutes)")
        print(f"Successful: {successful}")
        print(f"Failed:     {failed}")
        if successful + failed > 0:
            print(f"Avg time/record: {total_time/(successful+failed):.2f} seconds")
        print(f"Results saved to: {self.output_path}")
        print(f"{'='*50}")
    
    def check_ollama_status(self) -> bool:
        """Check if Ollama is running and the model is available."""
        try:
            response = requests.get(f"{self.base_url}/api/tags", timeout=5)
            response.raise_for_status()
            models = response.json().get("models", [])
            model_names = [m.get("name", "") for m in models]
            model_base_names = [m.split(":")[0] for m in model_names]
            
            print(f"Ollama is running. Available models: {model_names}")
            
            model_base = self.model_name.split(":")[0]
            if self.model_name in model_names or model_base in model_base_names:
                print(f"✓ Model '{self.model_name}' is available")
                if "qwen3" in self.model_name.lower():
                    print(f"  Note: Qwen3 thinking model detected - extended timeout set to {self.timeout}s")
                return True
            else:
                print(f"✗ Model '{self.model_name}' not found. Please run: ollama pull {self.model_name}")
                return False
        except requests.exceptions.RequestException as e:
            print(f"✗ Cannot connect to Ollama at {self.base_url}")
            print(f"  Error: {e}")
            print(f"  Make sure Ollama is running: ollama serve")
            return False


print("OllamaResumeGenerator class defined successfully!")

OllamaResumeGenerator class defined successfully!


In [ ]:
# Initialize the generator and check Ollama status
generator = OllamaResumeGenerator(
    model_name="qwen3:8b",  # Change to your preferred model
    output_path="dataset/tailored_resumes/tailored_resumes",
    timeout=180,
    max_retries=3
)

# Check if Ollama is running
generator.check_ollama_status()

Ollama is running. Available models: ['qwen3-vl:8b', 'qwen3:8b', 'phi4-reasoning:plus', 'gpt-oss:20b', 'deepseek-r1:14b', 'mixtral:8x7b', 'mistral-nemo:latest', 'mistral:7b']
✓ Model 'qwen3:8b' is available
  Note: Qwen3 thinking model detected - extended timeout set to 180s


True

In [30]:
# Process the dataset (start with a small batch to test)
# Uncomment the line below to process all records, or adjust start_idx and end_idx

# Test with first 5 records
generator.process_dataframe(
    df,
    resume_col="resume_text",
    job_col="job_description",
    filename_col="filename",
    start_idx=0,
    end_idx=1,  # Change to None to process all 1565 records
    save_every=1
)

Starting processing at: 2025-12-01T02:23:04.655779
Model: qwen3:8b
Thinking mode: Enabled
Output file: dataset\tailored_resumes_20251201_022302.jsonl
Processing records 0 to 1 (1 total)



Generating resumes: 100%|██████████| 1/1 [00:26<00:00, 26.10s/it]


Processing complete!
Model:      qwen3:8b
Started:    2025-12-01T02:23:04.655779
Finished:   2025-12-01T02:23:30.761733
Total time: 26.11 seconds (0.44 minutes)
Successful: 1
Failed:     0
Avg time/record: 26.11 seconds
Results saved to: dataset\tailored_resumes_20251201_022302.jsonl


In [35]:
# Read and display the generated results
results_df = pd.read_json('dataset\\tailored_resumes_20251201_022302.jsonl', lines=True)
print(f"Generated {len(results_df)} tailored resumes")
print(f"\nStatus distribution:")
print(results_df['status'].value_counts())

# Show a sample of a successful result
successful = results_df[results_df['status'] == 'success']
if len(successful) > 0:
    print(f"\n=== Sample Tailored Resume ===")
    sample = successful.iloc[0]
    print(f"Filename: {sample['filename']}")
    print(f"\nTailored Resume JSON:")
    print(json.dumps(sample['tailored_resume'], indent=2))

Generated 1 tailored resumes

Status distribution:
status
success    1
Name: count, dtype: int64

=== Sample Tailored Resume ===
Filename: 00_Willie_Ellis_Go_Python.docx

Tailored Resume JSON:
{
  "personal_information": {
    "name": "Willie Ellis",
    "email": "willieellis0177@gmail.com",
    "phone": "760 995 2578",
    "location": "Buffalo, New York",
    "socials": [
      {
        "name": "LinkedIn",
        "link": "http://www.linkedin.com/in/willieellisa6b492212"
      }
    ]
  },
  "summary": "Detail-oriented and dependable individual with a strong work ethic, capable of maintaining a safe and organized work environment. Eager to contribute to production goals and ensure quality standards.",
  "experiences": [
    {
      "designation": "Textiles Assistant",
      "companyName": "Goodwill Industries of Northwest NC",
      "location": "Brevard, NC",
      "start_date": "2023-08-01",
      "end_date": "Present",
      "points": [
        "Sorting clothing with attention to q

In [36]:
results_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 12 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   filename                 1 non-null      object        
 1   original_resume          1 non-null      object        
 2   job_description          1 non-null      object        
 3   status                   1 non-null      object        
 4   tailored_resume          1 non-null      object        
 5   raw_response             0 non-null      float64       
 6   thinking_content         0 non-null      float64       
 7   timestamp                1 non-null      datetime64[ns]
 8   processing_time_seconds  1 non-null      float64       
 9   original_index           1 non-null      int64         
 10  session_start            1 non-null      object        
 11  model_used               1 non-null      object        
dtypes: datetime64[ns](1), float64(3), int64(